# Week 5 — Cleaning Exploration

This notebook validates the silver cleaning layer.
We compare bronze vs silver outputs and verify each cleaning operation.

## Setup

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from src.orchestration import Pipeline

pipeline = Pipeline()
pipeline.run_bronze()
pipeline.run_silver()

## 1. Certifications Cleaning

### Before/After Comparison

In [ ]:
bronze = pipeline.bronze_certifications
silver = pipeline.silver_certifications
print(f"Bronze: {len(bronze):,} rows | Silver: {len(silver):,} rows")

# Verify sentinel DOB removed
sentinel_bronze = (pd.to_datetime(bronze["dob"]) == pd.Timestamp("1900-01-02")).sum()
sentinel_silver = (pd.to_datetime(silver["dob"], errors="coerce") == pd.Timestamp("1900-01-02")).sum()
print(f"Sentinel DOB (1900-01-02): bronze={sentinel_bronze}, silver={sentinel_silver}")

# Verify age=0 fix
age_zero_bronze = (bronze["age"] == 0).sum()
age_zero_silver = (silver["age"] == 0).sum()
print(f"Age = 0: bronze={age_zero_bronze}, silver={age_zero_silver}")

# Certificate booleans
cert_cols = ["mental_handicap", "parents_consent", "hap", "unified_partner"]
print(f"Certificate dtypes (silver): {silver[cert_cols].dtypes.to_dict()}")

## 2. Clubs Cleaning

### Country & Province Normalization

In [ ]:
bronze_clubs = pipeline.bronze_clubs
silver_clubs = pipeline.silver_clubs

print("Country values (bronze):", sorted(bronze_clubs["country"].dropna().unique())[:10], "...")
print("Country values (silver):", sorted(silver_clubs["country"].unique()))

print(f"Province values (silver): {sorted(silver_clubs['province'].dropna().unique())}")

# Verify Group 786 fix
g786 = silver_clubs[silver_clubs["group_number"] == 786]
print(f"Group 786: province={g786['province'].values[0]}, country={g786['country'].values[0]}")

# Verify zipcode fix
g787 = silver_clubs[silver_clubs["group_number"] == 787]
print(f"Group 787 zipcode: {g787['zipcode'].values[0]} (was 29900)")

## 3. Results Cleaning

### Gender Standardization

In [ ]:
bronze_res = pipeline.bronze_results
silver_res = pipeline.silver_results

print("Gender (bronze):", bronze_res["gender"].value_counts().to_dict())
print("Gender (silver):", silver_res["gender"].value_counts().to_dict())

### Place Parsing

In [ ]:
print(f"Rows with rank: {silver_res['rank'].notna().sum():,}")
print(f"Rank distribution:")
print(silver_res["rank"].value_counts().sort_index().head(10))
print(f"Result status distribution:")
print(silver_res["result_status"].value_counts().head(10))
print(f"Disqualified: {silver_res['is_disqualified'].sum():,}")

### Score Parsing

In [ ]:
print(f"Scores parsed: {silver_res['score_value'].notna().sum():,} / {silver_res['score'].notna().sum():,}")
print(f"Score unit distribution:")
print(silver_res["score_unit"].value_counts())

# Sample parsed scores by unit
for unit in ["seconds", "meters", "points"]:
    sample = silver_res[silver_res["score_unit"] == unit][["score", "score_value", "score_unit"]].head(3)
    print(f"{unit} samples:")
    print(sample.to_string())
    print()

### Multi-Round Deduplication

In [ ]:
print(f"Bronze results: {len(bronze_res):,} rows")
print(f"Silver results: {len(silver_res):,} rows")
removed = len(bronze_res) - len(silver_res)
print(f"Rows removed: {removed:,} ({removed / len(bronze_res) * 100:.1f}%)")

# Verify no duplicates remain
dupes = silver_res.duplicated(subset=["code", "event", "sport", "year"]).sum()
print(f"Remaining duplicates on (code, event, sport, year): {dupes} (should be 0)")
assert dupes == 0, "Duplicates should have been removed!"

### Per-Year Row Count Comparison

In [ ]:
comparison = pd.DataFrame({
    "bronze": bronze_res["year"].value_counts().sort_index(),
    "silver": silver_res["year"].value_counts().sort_index(),
})
comparison["removed"] = comparison["bronze"] - comparison["silver"]
comparison["removed_pct"] = (comparison["removed"] / comparison["bronze"] * 100).round(1)
print(comparison)

## Summary

| Table | Bronze | Silver | Delta | Key Changes |
|-------|--------|--------|-------|-------------|
| Certifications | 20,221 | 20,221 | 0 | DOB sentinels, age nulls, cert bools |
| Clubs | 436 | 436 | 0 | Country/province/city normalization |
| Results | 112,375 | ~72,700 | -35% | Gender, place/score parse, dedup |

All cleaning validated. Silver layer ready for Gold (Week 6).